In [ ]:
!pip install rotary-embedding-torch
#train_pos = 40248000
#val_pos  = 402480
#def get_batch(split):
#    global train_pos, val_pos
#
#    data = train_data if split == 'train' else val_data
#    pos = train_pos if split == 'train' else val_pos
#
#    x_list, y_list = [], []
#
#    for _ in range(batch):  # batch number of sequences
#        if pos + seq_len + 1 > len(data):  # wrap around if at the end
#            pos = 0
#
#        x = data[pos : pos + seq_len]
#        y = data[pos + 1 : pos + seq_len + 1]
#
#        x_list.append(torch.tensor(x, dtype=torch.long))
#        y_list.append(torch.tensor(y, dtype=torch.long))
#
#        pos += 1  # move forward linearly
#
#    # update position
#    if split == 'train':
#        train_pos = pos
#    else:
#        val_pos = pos
#
#    # stack sequences into batch
#    x_batch = torch.stack(x_list).to(device)
#    y_batch = torch.stack(y_list).to(device)
#    return x_batch, y_batch

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from rotary_embedding_torch import RotaryEmbedding
from transformers import GPT2TokenizerFast
import torch
import torch.nn as nn
import numpy as np
import os
import torch.nn.functional as F
tokenizer = GPT2TokenizerFast.from_pretrained('gpt2')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
seq_len = 512
batch = 16
# Add padding token if not present
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

vocab_size = len(tokenizer)
print(f"Vocabulary size: {vocab_size}")

def encode(text):
    """Encode text using GPT-2 tokenizer"""
    return tokenizer.encode(text, add_special_tokens=False)

def decode(indices):
    """Decode token indices back to text"""
    return tokenizer.decode(indices)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

Vocabulary size: 50257


In [ ]:
# use this if you want to tokenize a new npy file
chunk_size = 1_000_000
tokens = []
with open("/content/drive/MyDrive/Combined.txt","r",encoding="utf-8") as f:
        while True:
          chunk = f.read(chunk_size)
          if not chunk:
              break
          tokens.extend(encode(chunk))

tokens = np.array(tokens, dtype=np.int32)
print(f"Total tokens: {len(tokens)}")
np.save("tokens.npy", tokens)

In [ ]:
# use this if you want to tokenize a new bin file
import os
idx = 0
num_tokens = 2_000_000_000

tokens = np.memmap("tokens.bin", dtype=np.int32, mode="w+", shape=(num_tokens,))
with open(path + "/train_split.txt", "r",encoding='utf-8') as f:
    lines = []
    i = 0
    for line in f:
      lines.append(line)
      i += 1
      if i % 1000 == 0:
        lines = "".join(lines)
        if idx+len(lines) > num_tokens:
          break
        temp = tokenizer.encode(lines)
        tokens[idx : idx + len(temp)] = temp
        idx += len(temp)
        lines = []


tokens.flush()
print("process done")


In [ ]:
total_tokens = 2_000_000_000

# Load as a memory-mapped array
tokens = np.memmap(
    "/content/drive/MyDrive/tokens.bin",
    dtype=np.int32,
    mode="r",  # read-only
    shape=(total_tokens,)
)

# for this particular file
tokens = tokens[420:]
tokens = tokens[:-17004]

n = int(0.9*len(tokens))
train_data = tokens[:n]
val_data = tokens[n:]

def get_batch(split):
    data = train_data if split == 'train' else val_data
    max_start = len(data) - seq_len
    ix = torch.randint(0,max_start , (batch,))

    x = torch.stack([
        torch.from_numpy(data[i:i+seq_len])
        for i in ix
    ])

    y = torch.stack([
        torch.from_numpy(data[i+1:i+seq_len+1])
        for i in ix
    ])

    x = x.to(device=device, dtype=torch.long)
    y = y.to(device=device, dtype=torch.long)
    return x, y

In [ ]:
d_model = 768
heads = 8
n_layers = 8
drop = 0.1
class causal_self_attention(nn.Module):
  def __init__(self,d_model,Heads):
    super().__init__()
    assert d_model % Heads == 0, "d_model must be divisible by n_head"
    self.d_model = d_model
    self.n_head = Heads
    self.resid_dropout = nn.Dropout(drop)
    self.c_attn = nn.Linear(d_model, 3 * d_model, bias=False)
    self.c_proj = nn.Linear(d_model, d_model, bias=False)
    self.head_dim = d_model // Heads
    self.rotary = RotaryEmbedding(dim=self.head_dim, use_xpos = True )
  def forward(self,x):
    B,T,C = x.size()
    q, k, v  = self.c_attn(x).split(self.d_model, dim=2)
    k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)
    q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)
    v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)
    q, k = self.rotary.rotate_queries_and_keys(q, k)
    y = torch.nn.functional.scaled_dot_product_attention(q,k,v,is_causal=True,enable_gqa=True)
    y = y.transpose(1, 2).contiguous().view(B, T, C)
    y = self.resid_dropout(self.c_proj(y))
    return y


class TransformerBlock(nn.Module):
  def __init__(self,d_model, heads):
    super().__init__()
    self.norm1 = nn.RMSNorm(d_model)
    self.att = causal_self_attention(d_model,heads)
    self.norm2 = nn.RMSNorm(d_model)
    self.mlp = nn.Sequential(
        nn.Linear(d_model,d_model*4),
        nn.GELU(),
        nn.Linear(d_model*4,d_model),
        nn.Dropout(drop)
    )
  def forward(self,x):
   h = self.norm1(x)
   x = x + self.att(h)
   x = x + self.mlp(self.norm2(x))
   return x
class Transformer(nn.Module):
    def __init__(self, d_model, heads, n_layers):
        super().__init__()

        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, heads)
            for _ in range(n_layers)
        ])

        self.norm = nn.RMSNorm(d_model)

    def forward(self, x):
        for block in self.blocks:
            x = block(x)
        return self.norm(x)
class GPT(nn.Module):
  def __init__(self,vocab_size, d_model, heads, n_layers):
     super().__init__()
     self.token_emb = nn.Embedding(vocab_size,d_model)
     self.transformer = Transformer(d_model, heads, n_layers)
     self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
     self.lm_head.weight = self.token_emb.weight

  def forward(self, idx):

        # Token + position embeddings
        x = self.token_emb(idx)                 # (batch, seq_len, d_model)
        x = self.transformer(x)                 # (batch, seq_len, d_model)

        # Project to vocab
        logits = self.lm_head(x)                # (batch, seq_len, vocab_size)

        return logits
model = GPT(vocab_size, d_model, heads, n_layers)


def init_weights(module):
    if isinstance(module, nn.Linear):
        # Standard Xavier initialization for linear layers
        nn.init.xavier_uniform_(module.weight)
        if module.bias is not None:
            nn.init.zeros_(module.bias)
    elif isinstance(module, nn.Embedding):
        # Small normal initialization for embeddings
        nn.init.normal_(module.weight, mean=0.0, std=0.02)
    elif isinstance(module, nn.LayerNorm) or isinstance(module, nn.RMSNorm):
        # LayerNorm weights to 1, bias to 0
        nn.init.ones_(module.weight)

# Apply to the model
model.apply(init_weights)
m = model.to(device)

optimizer = torch.optim.AdamW(
    m.parameters(),
    lr=6e-4,
    weight_decay=0.1,
    betas=(0.9,0.95),
    fused=True
)

m = torch.compile(m)
criterion = nn.CrossEntropyLoss()

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=30000,
    eta_min=6e-5
)

In [ ]:
# loading the model
checkpoint = torch.load("/content/drive/MyDrive/checkpoint.pth",map_location=device)
m.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
scheduler.load_state_dict(checkpoint['scheduler_state_dict'])

In [ ]:
accum_steps = 8
def train_step():
    m.train()

    x, y = get_batch('train')           # x,y: (B, T)

    logits = m(x)                       # (B, T, vocab)
    B, T, V = logits.shape

    loss = criterion(
        logits.view(B*T, V),
        y.view(B*T)
    )

    loss = loss / accum_steps
    loss.backward()

    return loss.item()
@torch.no_grad()
def eval_step():
    m.eval()

    x, y = get_batch('val')
    logits = m(x)

    B, T, V = logits.shape
    loss = criterion(
        logits.view(B*T, V),
        y.view(B*T)
    )
    return loss.item()
max_iters = 100
eval_interval = 100

optimizer.zero_grad(set_to_none=True)
for step in range(max_iters):
    loss = train_step()
    if (step + 1) % accum_steps == 0:
        torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad(set_to_none=True)

    if step % 100 == 0:
        lr = optimizer.param_groups[0]['lr']
        print(f"step: {step} | lr: {lr:.2e} | train loss: {loss*accum_steps:.4f}")

    if step % eval_interval == 0:
        val_loss = eval_step()
        print(f"--- val loss: {val_loss:.4f} ---")

if max_iters % accum_steps != 0:
    torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
    optimizer.step()
    scheduler.step()
    optimizer.zero_grad(set_to_none=True)

checkpoint = {
    'model_state_dict': m.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'scheduler_state_dict': scheduler.state_dict(),
}
torch.save(checkpoint, "checkpoint.pth")


step: 0 | lr: 5.91e-04 | train loss: 1.8762
--- val loss: 2.5426 ---


In [ ]:
def apply_top_p(logits, top_p):
    # logits: (1, vocab)
    sorted_logits, sorted_indices = torch.sort(logits, descending=True)

    probs = torch.softmax(sorted_logits, dim=-1)
    cumulative_probs = torch.cumsum(probs, dim=-1)

    # remove tokens with cumulative prob above top_p
    cutoff = cumulative_probs > top_p

    # always keep at least 1 token
    cutoff[..., 1:] = cutoff[..., :-1].clone()
    cutoff[..., 0] = False

    # mask logits
    sorted_logits[cutoff] = -float('Inf')

    # unsort
    logits = torch.zeros_like(logits)
    logits.scatter_(1, sorted_indices, sorted_logits)

    return logits


In [ ]:
import time

@torch.no_grad()
def generate(model, idx, max_new_tokens, temperature=0.5, top_k=None,top_p=0.9,repetition_penalty=1.1):
    model.eval()
    idx = idx.to(next(model.parameters()).device)
    for _ in range(max_new_tokens):
        # crop idx to last seq_len tokens to fit model input
        idx_cond = idx[:, -seq_len:]
        # forward pass
        logits = m(idx_cond)              # (1, seq_len, vocab_size)
        logits = logits[:, -1, :]             # (1, vocab) -> last token

        # apply temperature
        logits = logits / temperature

        if repetition_penalty != 1.0:
          for token_id in set(idx[0].tolist()):
              if logits[0, token_id] > 0:
                  logits[0, token_id] /= repetition_penalty
              else:
                  logits[0, token_id] *= repetition_penalty

        # optionally restrict to top_k
        if top_k is not None:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = -float('Inf')

        if top_p is not None:
            logits = apply_top_p(logits, top_p)


        # convert to probabilities
        probs = F.softmax(logits, dim=-1)

        # sample next token
        next_idx = torch.multinomial(probs, num_samples=1)
        idx = torch.cat([idx, next_idx], dim=1)
    return idx

# starting token(s)
start_text = 'What is'
start_tokens = torch.tensor([encode(start_text)], dtype=torch.long)

generated_tokens = generate(m, start_tokens, max_new_tokens=50, temperature=0.75, top_k=None,top_p=0.9,repetition_penalty=1.1)

generated_text = decode(generated_tokens[0].tolist())

words = generated_text.split(" ")
for w in words:
    print(w + " ", end="", flush=True)
    time.sleep(0.02 if w != "\n" else 0.2)

What is so much more.

I think I can’t give you any words about how things are going in the face of life, but I can say that it’s hard to find. You can tell me that it’s 